In [1]:
from pathlib import Path
import os,json,hashlib
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
import numpy as np,h5py,torch
from torch import nn
from sklearn.metrics import roc_auc_score,average_precision_score
ROOT=next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p/'dataset_2.csv').is_file());RUN=ROOT/'bilstm_eew/bilstm_training_v1';OUT=RUN/'codex_independent_audit';DATA=ROOT/'bilstm_eew/causal_tabular_v1/tabular_features.h5';CNN=ROOT/'cnn/cnn wave/eew_causal_v3/cnn_eew_training/codex_performance_audit/cnn_performance_audit_results.json'
DEV=torch.device('cuda' if torch.cuda.is_available() else 'cpu');torch.set_num_threads(4);torch.use_deterministic_algorithms(True);torch.backends.cudnn.benchmark=False
def sha(p):
 h=hashlib.sha256()
 with p.open('rb') as f:
  for b in iter(lambda:f.read(4194304),b''):h.update(b)
 return h.hexdigest()
paths=[DATA,CNN,RUN/'final_bilstm_eew_report.md',RUN/'01_bilstm_training.ipynb',RUN/'02_bilstm_evaluation.ipynb',ROOT/'bilstm_eew/causal_tabular_v1/feature_validation_report.md']+[RUN/'models'/f'bilstm_{h}s_best.pt' for h in [1,2,3]]
before={str(p):sha(p) for p in paths};text=(RUN/'final_bilstm_eew_report.md').read_text(encoding='utf-8');reported=json.loads(text.split('```json\n')[1].split('\n```')[0]);cnn=json.loads(CNN.read_text(encoding='utf-8'))['common_cohort']
R={'checks':{},'horizons':{},'input_hashes':before}
def check(name,ok,evidence):R['checks'][name]={'status':'PASS' if ok else 'FAIL','evidence':evidence}
# Independent reconstruction, no original executable cells run.
class Model(nn.Module):
 def __init__(self):
  super().__init__();self.lstm=nn.LSTM(7,32,2,batch_first=True,bidirectional=True,dropout=.2);self.shared=nn.Sequential(nn.Dropout(.2),nn.Linear(64,32),nn.SiLU());self.detect=nn.Linear(32,1);self.mag=nn.Linear(32,1)
 def forward(self,x):
  o,_=self.lstm(x);o=self.shared(o.mean(dim=1));return self.detect(o).flatten(),self.mag(o).flatten()
@torch.inference_mode()
def predict(model,x):
 a=[];b=[]
 for start in range(0,len(x),256):
  u,v=model(torch.from_numpy(x[start:start+256]).to(DEV));a.append(u.cpu());b.append(v.cpu())
 return torch.cat(a),torch.cat(b)
def det(y,p,t):
 y=np.asarray(y)==1;q=p>=t;tp=int((y&q).sum());fp=int((~y&q).sum());fn=int((y&~q).sum());tn=int((~y&~q).sum())
 return dict(accuracy=(tp+tn)/len(y),precision=tp/(tp+fp) if tp+fp else 0.,recall=tp/(tp+fn),f1=2*tp/(2*tp+fp+fn),false_positive_rate=fp/(fp+tn),roc_auc=float(roc_auc_score(y,p)),pr_auc_ap=float(average_precision_score(y,p)),confusion_matrix=[[tn,fp],[fn,tp]])
def mag(a,p):
 a=a.astype(float);p=p.astype(float);e=p-a
 return dict(mae=float(np.abs(e).mean()),rmse=float(np.sqrt(np.mean(e**2))),r2=float(1-np.sum(e**2)/np.sum((a-a.mean())**2)),bias=float(e.mean()),actual_variance=float(a.var()),predicted_variance=float(p.var()),std_ratio=float(p.std()/a.std()),slope=float(np.polyfit(a,p,1)[0]))
ids={}
with h5py.File(DATA,'r') as f:
 for split in ['train','validation','test']:
  g=f[split];ids[split]={k:set(g[k].asstr()[:]) for k in ['source_id','parent_id']}
 for h in [1,2,3]:
  ck=torch.load(RUN/'models'/f'bilstm_{h}s_best.pt',map_location='cpu',weights_only=False);model=Model().to(DEV);model.load_state_dict(ck['state_dict'],strict=True);model.eval()
  check(f'{h}s_checkpoint',ck['horizon']==h and all(before.get(p)==v for p,v in ck['input_hashes'].items()),{'path':str(RUN/'models'/f'bilstm_{h}s_best.pt'),'sha256':before[str(RUN/'models'/f'bilstm_{h}s_best.pt')]})
  for split in ['train','validation','test']:
   g=f[split];y=g['labels'][:];keep=g[f'classification_mask_{h}s'][:];reg=g[f'magnitude_mask_{h}s'][:];pre=g[f'pre_s_valid_{h}s'][:]
   check(f'{split}_{h}s_masks',np.array_equal(reg,(y==1)&keep) and np.array_equal(reg,pre) and np.array_equal(keep,(y==0)|pre) and np.all(g['magnitude'][:][y==0]==-1) and g[f'{h}s/X'].shape[1:]==(10+10*h,7),'Existing pre-S masks reused; only eligible rows supplied; no beyond-cutoff feature bins. Original arrival/feature audits not repeated.')
  sg=f[f'scalers/{h}s'];check(f'{h}s_feature_scaler',sg.attrs['fit_split']=='train' and np.array_equal(sg['mean'][:],ck['feature_scaler']['mean']) and np.array_equal(sg['scale'][:],ck['feature_scaler']['scale']) and sg.attrs['fit_records']==int(f[f'train/classification_mask_{h}s'][:].sum()),'Checkpoint parameters exactly match approved HDF5 train-only scaler; no refit in training code. Reliance on prior feature validation explicitly retained.')
  tm=f['train/magnitude'][:][f[f'train/magnitude_mask_{h}s'][:]];center=float(torch.from_numpy(tm).mean());scale=float(torch.from_numpy(tm).std(unbiased=False))
  check(f'{h}s_target_scaler',center==ck['center'] and scale==ck['scale'],{'training_mean':center,'training_sd':scale})
  g=f['validation'];keep=g[f'classification_mask_{h}s'][:];reg=g[f'magnitude_mask_{h}s'][:][keep];y=g['labels'][:][keep];m=g['magnitude'][:][keep]
  vl,vz=predict(model,g[f'{h}s/X'][:][keep]);vp=vl.sigmoid().numpy();grid=np.round(np.arange(.10,.901,.01),2);scores=[det(y,vp,t)['f1'] for t in grid];j=max(range(len(grid)),key=lambda i:(scores[i],-abs(grid[i]-.5),-grid[i]));reselected=float(grid[j])
  loss=float(nn.functional.binary_cross_entropy_with_logits(vl,torch.tensor(y,dtype=torch.float32)))+float(nn.functional.huber_loss(vz[reg],(torch.tensor(m)[reg]-center)/scale,delta=1.))
  best=float('inf');best_epoch=None;stale=0
  for item in ck['history']:
   if item['validation']['total']<best-ck['config']['min_delta']:best=item['validation']['total'];best_epoch=item['epoch'];stale=0
   else:stale+=1
  check(f'{h}s_selection',best_epoch==ck['best_epoch'] and abs(loss-ck['best_validation_loss'])<1e-6 and reselected==ck['threshold'],{'epoch':best_epoch,'validation_loss_delta':loss-ck['best_validation_loss'],'saved_threshold':ck['threshold'],'validation_reproduced_threshold':reselected,'validation_f1':scores[j]})
  g=f['test'];keep=g[f'classification_mask_{h}s'][:];reg=g[f'magnitude_mask_{h}s'][:][keep];common=g['common_cohort_mask'][:][keep];y=g['labels'][:][keep];m=g['magnitude'][:][keep]
  logits,z=predict(model,g[f'{h}s/X'][:][keep]);p=logits.sigmoid().numpy();pred=z.numpy()*scale+center
  digest=hashlib.sha256('\n'.join(g['source_id'].asstr()[:][keep][common]).encode()).hexdigest()
  check(f'{h}s_common_cohort',int((reg&common).sum())==1200 and int((~reg&common).sum())==1214 and digest==cnn['identity_sha256'],{'earthquakes':int((reg&common).sum()),'noise':int((~reg&common).sum()),'ordered_source_id_sha256':digest,'ordered_parent_id_sha256':hashlib.sha256('\n'.join(g['parent_id'].asstr()[:][keep][common]).encode()).hexdigest()})
  v={'threshold':ck['threshold'],'epoch':best_epoch,'n':len(y),'detection':det(y,p,ck['threshold']),'fixed_05_detection':det(y,p,.5),'magnitude':mag(m[reg],pred[reg]),'common':{'detection':det(y[common],p[common],ck['threshold']),'fixed_05_detection':det(y[common],p[common],.5),'magnitude':mag(m[reg&common],pred[reg&common])}}
  diffs={}
  for name in ['detection','fixed_05_detection','magnitude']:
   for key,value in v[name].items():
    if key in reported[str(h)][name] and key!='confusion_matrix':diffs[name+'/'+key]=value-reported[str(h)][name][key]
   if name!='magnitude':check(f'{h}s_{name}_confusion',v[name]['confusion_matrix']==reported[str(h)][name]['confusion_matrix'],v[name]['confusion_matrix'])
  for name in ['detection','fixed_05_detection','magnitude']:
   for key,value in v['common'][name].items():
    if key in reported[str(h)]['common'][name] and key!='confusion_matrix':diffs['common/'+name+'/'+key]=value-reported[str(h)]['common'][name][key]
  a=m[reg].astype(float);b=pred[reg].astype(float);strong=a>=5;e=b[strong]-a[strong]
  v['strong']={'n':int(strong.sum()),'underprediction_percent':float((e<0).mean()*100),'mae':float(np.abs(e).mean()),'bias':float(e.mean())}
  expected=reported[str(h)]['bands']['M>=5'];check(f'{h}s_strong_bias',v['strong']['n']==expected['n'] and abs(v['strong']['underprediction_percent']-expected['underprediction_percent'])<1e-10 and abs(v['strong']['mae']-expected['mae'])<1e-6 and abs(v['strong']['bias']-expected['bias'])<1e-6,v['strong'])
  v['discrepancies']=diffs;R['horizons'][h]=v;check(f'{h}s_full_precision_metrics',max(abs(x) for x in diffs.values())<1e-6,{'max_abs_difference':max(abs(x) for x in diffs.values()),'tolerance':1e-6})
  print(f'{h}s verified: epoch {best_epoch}, threshold {reselected:.2f}, F1 {v["detection"]["f1"]:.6f}, MAE {v["magnitude"]["mae"]:.6f}')
for a,b in [('train','validation'),('train','test'),('validation','test')]:check(a+'/'+b+'_identity_isolation',all(not(ids[a][k]&ids[b][k]) for k in ids[a]),'No source/parent identity overlap in feature metadata.')
check('saved_protocol_no_test_tuning',True,'Training notebook optimization uses train; epoch/threshold selection uses validation only. Test inference appears only after all three frozen checkpoints exist. Grid/tie rule reproduced independently; no new thresholds adopted. Target metadata is outside model.forward. This verifies artifacts and saved code, not unrecorded historical actions.')
R['checks']['historical_execution_certainty']={'status':'UNKNOWN','evidence':'No immutable external execution log. Existing test set was already used in prior CNN research, so this is exploratory comparison rather than fresh confirmatory evaluation.'}
R['checks']['noise_family_shortcuts']={'status':'UNKNOWN','evidence':'Feature file lacks noise-family labels. No labels fabricated; upstream dataset/feature validation not rerun.'}
check('preservation',all(sha(Path(p))==v for p,v in before.items()),{'files':len(before),'changed':[] if all(sha(Path(p))==v for p,v in before.items()) else 'See hash failure'})
R['verdict']='BLOCKED' if any(c['status']=='FAIL' for c in R['checks'].values()) else 'APPROVED'
R['scope']='Verified standalone synthetic benchmark reporting and controlled comparison; suitable baseline for a separately designed fusion experiment, not evidence that fusion improves performance or authorization to begin it.'
with (OUT/'audit_results.json').open('x',encoding='utf-8') as f:json.dump(R,f,indent=2,allow_nan=False)
lines=['# Independent BiLSTM performance audit','', '**Date:** 2026-11-13. **Verdict: '+R['verdict']+'**','',R['scope'],'','## Verification','','Independent model reconstruction and read-only frozen checkpoint inference on validation/test; no training or original notebook execution. Exact numerical report JSON reused as comparison values, not accepted as proof. No standalone prediction archive existed. Feature HDF5 was read only as the required inference/mask/scaler dependency. The prior feature-extraction validation is reused, not repeated.','', '| Check | Status | Evidence |','|---|---|---|']
for k,c in R['checks'].items():lines.append('| '+k+' | '+c['status']+' | '+str(c['evidence']).replace('|','/')+' |')
lines+=['','## Horizon-eligible detection at both thresholds','','FPR=FP/(FP+TN). Magnitude metrics do not condition on successful detection.','', '| Horizon | Threshold | Precision | Recall | F1 | FPR | Accuracy |','|---|---:|---:|---:|---:|---:|---:|']
for h,v in R['horizons'].items():
 for t,key in [(.5,'fixed_05_detection'),(v['threshold'],'detection')]:
  d=v[key];lines.append(f'| {h}s | {t:.2f} | '+' | '.join(f'{d[k]:.8f}' for k in ['precision','recall','f1','false_positive_rate','accuracy'])+' |')
lines+=['','## Magnitude and compression','','| Horizon | MAE | RMSE | R2 | Pred/actual SD | M>=5 N | Underprediction % | Strong bias |','|---|---:|---:|---:|---:|---:|---:|---:|']
for h,v in R['horizons'].items():
 m=v['magnitude'];b=v['strong'];lines.append(f"| {h}s | {m['mae']:.9f} | {m['rmse']:.9f} | {m['r2']:.9f} | {m['std_ratio']:.5f} | {b['n']} | {b['underprediction_percent']:.5f} | {b['bias']:.6f} |")
lines+=['','## Identical CNN common cohort','','Each horizon matches the audited ordered source-ID hash for exactly 1,200 earthquakes and 1,214 noise records. Parent-ID hashes are recorded too; source-parent association relies on unchanged validated feature provenance, not a rerun of waveform joins.','', '| Horizon | CNN F1 .5 | BiLSTM F1 .5 | BiLSTM F1 selected | CNN MAE | BiLSTM MAE |','|---|---:|---:|---:|---:|---:|']
for h,v in R['horizons'].items():
 c=cnn['metrics'][str(h)];a=v['common'];lines.append(f"| {h}s | {c['f1']:.8f} | {a['fixed_05_detection']['f1']:.8f} | {a['detection']['f1']:.8f} | {c['mae']:.8f} | {a['magnitude']['mae']:.8f} |")
lines+=['','Full precision metrics, signed differences versus saved results, confusion matrices and both threshold policies on the common cohort are in audit_results.json. Differences up to 1e-6 are treated as numerical precision; exact threshold choices and confusion counts must agree.','', '## Interpretation and later fusion boundaries','','All reported numbers are valid synthetic baseline results if the checks above pass; weak performance is not an evaluation-integrity failure. Validation-F1 selection produces high false-positive rates (approximately 66–74%). Strong-event underprediction and compressed variance persist. The BiLSTM is not uniformly superior to CNN: threshold policies must be matched, and tiny magnitude differences are not significance evidence. Feature representation, model capacity and budget also differ.','', 'No test-driven optimization is present in saved code, and validation threshold selection reproduces without test labels. This cannot prove absence of unrecorded manual experimentation. The shared test cohort has already been viewed and must not guide future fusion architecture, calibration, threshold or checkpoint selection. For a later authorized fusion experiment, use existing train/validation roles, identical event/parent alignment, per-horizon masks and train-only transforms; define the protocol before any further test evaluation. Fresh evidence would be needed for strong confirmatory claims. No fusion was implemented or begun.','', 'Noise-family shortcut status remains UNKNOWN with this feature-only interface. Both branches originate from the same synthetic observations; their errors may be correlated. No complementary information or fusion gain has been established. Operational trigger capability, real-data validity and warning lead time remain unverified. Bidirectional processing covers only the observed 20/30/40 bins. Stored pre-S masks are offline eligibility metadata, not model features or deployable gates.','', 'Minor documentation issue: the source Markdown contains encoding artifacts in some symbols. The embedded numerical JSON is readable and metrics are unaffected; no protected report was repaired.','', '## Evidence and preservation','']
for p,v in before.items():lines.append(f'- `{p}` — SHA256 `{v}`')
lines+=['', 'Only audit.ipynb, audit_report.md and audit_results.json created. No retraining, tuning, new thresholds, dataset regeneration, figures, cleanup or changes to existing files.']
with (OUT/'audit_report.md').open('x',encoding='utf-8') as f:f.write('\n'.join(lines)+'\n')
print('VERDICT',R['verdict'],'FAILURES',[k for k,c in R['checks'].items() if c['status']=='FAIL'])


1s verified: epoch 18, threshold 0.47, F1 0.729405, MAE 0.678042
2s verified: epoch 27, threshold 0.36, F1 0.728481, MAE 0.638256
3s verified: epoch 26, threshold 0.36, F1 0.731584, MAE 0.615082
VERDICT APPROVED FAILURES []
